In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.cross_decomposition import PLSRegression

In [2]:
z_score_df = pd.read_pickle('matrix_Coimbra_threshold.pkl')

In [3]:
z_score_df

Protein.Group,A0A075B6H7,A0A075B6K5,A0A075B6P5;P01615;A0A087WW87;P01614,A0A075B6S5;A0A0C4DH67;A0A0C4DH69,A0A0A0MRZ8;P04433,A0A0A0MS15,A0A0B4J1U7,A0A0B4J1X8,A0A0B4J1Y9,A0A0B4J2B5,...,Q9UBP4,Q9UBX5,Q9UKI9;P09086;P14859,Q9ULB1,Q9Y4C0,Q9Y646,Q9Y6R7,cRAP-P00760,cRAP-P0AEY0,cRAP-P42212
109094,-0.235900,0.368910,-0.001399,-0.250872,-0.547748,-1.042582,-0.160501,-0.143467,0.005032,-0.948622,...,-0.414465,-1.411198,-0.603788,0.279622,-0.086622,-0.504171,-0.679232,-1.199426,-0.533962,-0.300272
103208,-0.477700,-0.881979,-0.168997,-0.985325,-0.561344,0.057938,-1.014755,-0.658864,-0.468966,-0.312457,...,-0.327057,-0.636241,0.490588,-0.484750,-0.389092,0.438867,-0.503038,-0.962731,-0.401852,-0.413514
106086,0.111044,0.284163,0.250087,-0.378412,-0.227504,-0.301630,-0.503469,-0.126363,0.518555,-0.118962,...,0.571070,-0.469992,0.153253,1.518716,1.494299,1.107409,0.174871,-0.447917,-0.783628,-0.764881
105634,0.990569,1.625988,1.662398,0.913341,1.220554,0.855728,-1.121883,0.649082,0.438595,1.152746,...,0.852665,-0.976623,0.658613,-1.926108,-0.802330,-0.189145,-1.002443,-0.748056,-1.934922,-1.830716
106008,-0.204214,-0.317213,0.782627,0.151888,0.002889,-1.165807,0.532662,0.500921,0.245322,0.025983,...,1.232430,-0.041504,0.556811,0.592066,-0.567734,-0.174536,0.419020,0.335909,-0.934451,-0.797986
110203,-0.802347,-0.716040,-0.989515,-0.617416,-0.383789,-0.380295,0.942366,-0.452439,-0.674361,-1.005461,...,1.351128,-0.721190,-0.218162,1.013610,0.867521,-0.380777,-0.600125,1.152790,0.639728,0.906787
108382,0.285327,0.618069,0.324381,-0.185258,-0.651542,-0.048090,1.203597,-0.527076,0.215549,-0.390191,...,0.301279,-0.769686,0.267671,-0.659311,-0.598041,0.530081,0.950284,-0.515290,-0.955682,-0.508277
104968,-0.237161,0.360176,0.078879,-0.085830,-0.863270,-0.230325,0.556989,0.287000,-0.682868,-0.124799,...,-0.248154,-1.584071,-0.278365,0.214096,-0.240083,1.064027,1.397432,-0.033633,0.046453,-0.029472
107018,-0.698645,0.635296,0.293461,-0.506870,-0.354595,-0.043427,0.835735,0.361479,0.456846,0.267781,...,-0.716436,-0.847352,0.546170,0.055156,0.131578,-1.434655,-1.457332,-0.263177,0.694673,0.681358
109456,-1.728946,-2.005236,-1.509203,-1.384965,-1.411332,-2.031558,-2.276153,-1.226211,-1.741858,-1.582536,...,0.540750,-0.422296,-0.586496,0.932252,0.614257,-0.188065,-1.702706,-0.344019,0.425366,0.530735


In [4]:
import pickle

with open('list_groups_Coimbra.pkl', 'rb') as f:
    list_groups = pickle.load(f)

print(list_groups)

['MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-CT', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD', 'MCI-AD']


In [5]:
list_groups = pd.Series(list_groups)

In [6]:
def calculate_vip(pls, X):
    
    t = pls.x_scores_
    w = pls.x_weights_
    q = pls.y_loadings_
    
    p, h = w.shape
    s = np.diag(t.T @ t @ q.T @ q).reshape(h, -1)
    
    Wnorm = (w ** 2) / np.sum(w ** 2, axis=0)
    
    vip = np.sqrt(p * (Wnorm @ s) / np.sum(s))
    
    return vip.flatten()

In [7]:
def stratified_bootstrap(X, y):
    """
    Creates a bootstrap sample preserving the original class proportions.
    """
    ad_idx = np.where(y == "MCI-AD")[0]
    ct_idx = np.where(y == "MCI-CT")[0]

    ad_sample = np.random.choice(ad_idx, size=len(ad_idx), replace=True)
    ct_sample = np.random.choice(ct_idx, size=len(ct_idx), replace=True)

    indices = np.concatenate([ad_sample, ct_sample])
    np.random.shuffle(indices)

    return X.iloc[indices], y.iloc[indices]


# Settings
n_iterations = 100
feature_names = z_score_df.columns

importance_matrix = np.zeros((n_iterations, len(feature_names)))
ranking_matrix = np.zeros((n_iterations, len(feature_names)))

vip_matrix = np.zeros((n_iterations, len(feature_names)))
vip_ranking_matrix = np.zeros((n_iterations, len(feature_names)))
vip_selection_matrix = np.zeros((n_iterations, len(feature_names)))

le = LabelEncoder()
le.fit(list_groups)


# Main loop
for i in range(n_iterations):

    X_boot, y_boot = stratified_bootstrap(z_score_df, list_groups)

    # =====================
    # RANDOM FOREST 
    # =====================

    rf = RandomForestClassifier(
        n_estimators=1000,
        class_weight="balanced",
        n_jobs=-1,
        random_state=i
    )

    rf.fit(X_boot, y_boot)

    # importances
    importances = rf.feature_importances_
    importance_matrix[i] = importances

    # ranking (1 = migliore)
    ranks = (-importances).argsort().argsort() + 1
    ranking_matrix[i] = ranks

    # =====================
    # PLS-DA 
    # =====================

    # encode y
    y_encoded = le.transform(y_boot)

    pls = PLSRegression(n_components=2)
    pls.fit(X_boot, y_encoded)

    # VIP
    vip_scores = calculate_vip(pls, X_boot)
    vip_matrix[i] = vip_scores

    # ranking VIP (1 = migliore)
    vip_ranks = (-vip_scores).argsort().argsort() + 1
    vip_ranking_matrix[i] = vip_ranks
    vip_selection_matrix[i] = (vip_scores > 1).astype(int)


# Aggregazione risultati
mean_importance = importance_matrix.mean(axis=0)
mean_rank = ranking_matrix.mean(axis=0)
std_rank = ranking_matrix.std(axis=0)

mean_vip = vip_matrix.mean(axis=0)
mean_vip_rank = vip_ranking_matrix.mean(axis=0)
std_vip_rank = vip_ranking_matrix.std(axis=0)
vip_freq = vip_selection_matrix.mean(axis=0)


# DataFrame finale
importance_df = pd.DataFrame({
    "protein": feature_names,
    "importance": mean_importance,
    "mean_rank": mean_rank,
    "rank_std": std_rank
}).sort_values(by="importance", ascending=False)

vip_df = pd.DataFrame({
    "protein": feature_names,
    "VIP": mean_vip,
    "mean_rank_vip": mean_vip_rank,
    "std_rank_vip": std_vip_rank,
    "vip_freq": vip_freq
}).sort_values(by="VIP", ascending=False)

In [8]:
vip_df

,protein,VIP,mean_rank_vip,std_rank_vip,vip_freq
176,P61916,2.217911,1.20,0.489898,1.00
70,P02751,2.086182,2.90,1.734935,1.00
110,P07998,2.007868,4.86,2.993393,1.00
68,P02749,1.984973,5.07,2.458679,1.00
139,P18065,1.927461,7.27,4.061662,1.00
...,...,...,...,...,...
186,Q13332,0.365606,197.40,40.126301,0.01
122,P0C0L5,0.365211,204.19,32.801431,0.01
179,P98160,0.343418,204.91,39.877837,0.02
124,P10451,0.326502,206.29,33.250653,0.00


In [9]:
importance_df

,protein,importance,mean_rank,rank_std
176,P61916,0.059942,1.93,1.305795
70,P02751,0.046582,4.12,3.105737
68,P02749,0.046515,4.21,2.854102
139,P18065,0.037369,7.33,5.265083
110,P07998,0.035253,7.92,5.143306
...,...,...,...,...
62,P02675,0.000299,194.28,33.817179
7,A0A0B4J1X8,0.000286,196.34,31.975997
50,P01780,0.000283,196.33,31.677454
99,P06681,0.000280,197.12,34.731910


In [14]:
final_df = importance_df.merge(vip_df, on="protein")
final_df["rank_diff"] = abs(final_df["mean_rank"] - final_df["mean_rank_vip"])
final_df["mean_rank_combined"] = (
    final_df["mean_rank"] + final_df["mean_rank_vip"]
) / 2
final_df["std_rf"] = final_df["rank_std"]
final_df["std_pls"] = final_df["std_rank_vip"]
final_df = final_df[[
    "protein",
    "mean_rank_combined",
    "std_rf",
    "std_pls",
    "vip_freq",
    "rank_diff"
]]
final_df = final_df.sort_values(by="mean_rank_combined")

In [15]:
final_df.head(20)

,protein,mean_rank_combined,std_rf,std_pls,vip_freq,rank_diff
0,P61916,1.565,1.305795,0.489898,1.00,0.73
1,P02751,3.510,3.105737,1.734935,1.00,1.22
2,P02749,4.640,2.854102,2.458679,1.00,0.86
4,P07998,6.390,5.143306,2.993393,1.00,3.06
3,P18065,7.300,5.265083,4.061662,1.00,0.06
5,P49641,10.265,6.078520,5.030417,1.00,0.39
6,P02766,10.480,6.420156,4.853205,1.00,1.12
7,Q12805,11.495,13.017200,8.231428,1.00,5.11
10,Q92876,13.745,9.633789,6.766062,1.00,3.53
9,Q9UBX5,14.235,8.611498,7.512683,1.00,0.75
